# CouchDB: load, verify and query the Grab Mobility & Fulfillment database

This notebook runs the CouchDB implementation end to end (report sections 2.3, 3.1 and 3.2):

1. connect to CouchDB and show the environment,
2. *(optional)* load the canonical dataset into CouchDB,
3. verify that CouchDB holds exactly the canonical rows,
4. run each of the 25 essential operations once and check its result,
5. query the database freely (documents, views, Mango),
6. *(optional)* run the benchmark (1 warm-up + 3 measured runs per operation).

**Before running**

```bash
cp .env.example .env            # then set the passwords
docker compose --env-file .env -f docker/compose.yaml up -d --build
uv sync --frozen
```

Open this notebook with the project's `.venv` kernel and run the cells from top to bottom.

## Parameters

* `DATA_DIR`: the dataset to load and check against. `data/canonical` is the full dataset
  (2,000,000 orders). For a quick trial on real CouchDB, any folder with the same 13 CSV
  files works, e.g. a sample of every 50th order.
* `RUN_LOAD = True` **drops** the four `grab_*` databases and loads `DATA_DIR` again. For the full
  dataset this takes a while (loading plus building the view indexes).
* `RUN_BENCHMARK = True` runs the benchmark runner over the 25 operations.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / 'data' / 'canonical'
RUN_LOAD = False
RUN_BENCHMARK = False
PREVIEW_ROWS = 10

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

## 1. Connect

In [ ]:
from couchdb import design, load, reference, setup_db, verify_load
from couchdb.client import CouchClient
from couchdb.queries import OPERATIONS, GrabCouch
from couchdb.settings import DATABASES, RESULTS_DIR, get_settings

settings = get_settings()
admin = CouchClient.admin(settings, timeout=None)
app = CouchClient.app(settings, timeout=None)
grab = GrabCouch(app, settings)
DB = {name: settings.db(name) for name in DATABASES}

server = setup_db.wait_until_up(admin)
versions = admin.json('GET', '/_node/_local/_versions')
print(f"CouchDB {server.get('version')} at {settings.url}")
print(f"JavaScript engine: {versions.get('javascript_engine')}")
print(f"Operations run as: {app.user} (admin: {admin.user})")

In [ ]:
def database_overview():
    rows = []
    for name, db in DB.items():
        if not admin.db_exists(db):
            rows.append({'database': db, 'exists': False})
            continue
        info = admin.db_info(db)
        sizes = info.get('sizes') or {}
        rows.append({'database': db, 'exists': True, 'documents': info['doc_count'],
                     'deleted': info['doc_del_count'],
                     'file_MB': round(sizes.get('file', 0) / 2**20, 1),
                     'data_MB': round(sizes.get('external', 0) / 2**20, 1),
                     'part2_tables': ', '.join(DATABASES[name])})
    return pd.DataFrame(rows)

database_overview()

## 2. Load the dataset (optional)

In [ ]:
LOAD_REPORT = RESULTS_DIR / 'couchdb_load_report.json'
if RUN_LOAD:
    exit_code = load.main(['--data-dir', str(DATA_DIR), '--reset', '--report', str(LOAD_REPORT)])
    print('load exit code:', exit_code)

if LOAD_REPORT.is_file():
    report = json.loads(LOAD_REPORT.read_text())
    print(f"Loaded from {report['data_dir']} with CouchDB {report['couchdb_version']}, "
          f"batch {report['batch_size']}, {report['workers']} workers")
    display(pd.DataFrame([{'database': db, **v} for db, v in report['databases'].items()]))
    display(pd.DataFrame(report.get('index_builds', [])))
else:
    print('No load report yet: set RUN_LOAD = True to load DATA_DIR.')

## 3. Verify the load (section 3.1)

Every Part 2 table must have the same number of rows in CouchDB (counted by the
`entity_counts` views, e.g. trips embedded in order documents), and a random sample of
documents must turn back into exactly the original CSV rows.

In [ ]:
VERIFY_REPORT = RESULTS_DIR / 'couchdb_verify_report.json'
verify_load.main(['--data-dir', str(DATA_DIR), '--sample', '1000', '--report', str(VERIFY_REPORT)])
verification = json.loads(VERIFY_REPORT.read_text())
display(pd.DataFrame(verification['tables']))
pd.DataFrame([{'entity': k, 'sampled': v['sampled'], 'mismatches': len(v['mismatches'])}
              for k, v in verification['samples'].items()])

## 4. Run the essential operations (section 3.2)

Expected results are computed straight from the CSV files of `DATA_DIR`
(`couchdb/reference.py`, which re-implements the Part 2 SQL), and every result is compared
with them. Write operations prepare their starting state, run, are checked by reading the
documents back, and are then undone, so the database ends as it started.

In [ ]:
ref_name = 'couchdb_reference.json' if DATA_DIR.name == 'canonical' else f'couchdb_reference_{DATA_DIR.name}.json'
REFERENCE_FILE = ROOT / 'data' / 'reference' / ref_name
os.environ['COUCHDB_REFERENCE_FILE'] = str(REFERENCE_FILE)
stale = (not REFERENCE_FILE.is_file()
         or json.loads(REFERENCE_FILE.read_text())['meta']['data_dir'] != str(DATA_DIR))
if stale:
    reference.main(['--data-dir', str(DATA_DIR), '--output', str(REFERENCE_FILE)])
else:
    print(f'Using expected results in {REFERENCE_FILE}')

In [ ]:
from benchmarks.adapters import couchdb as adapter

SCENARIOS = {s.operation_id: s for s in adapter.create_scenarios()}
META = {op.operation_id: op for op in OPERATIONS}
SUMMARY = {}

def as_table(result, rows=PREVIEW_ROWS):
    if isinstance(result, list):
        return pd.DataFrame(result[:rows])
    if isinstance(result, dict) and result.get('type') == 'service_order':
        return Markdown('```json\n' + json.dumps(result, indent=1)[:4000] + '\n```')
    return Markdown('```json\n' + json.dumps(result, indent=1, default=str)[:3000] + '\n```')

def run(operation_id, rows=PREVIEW_ROWS):
    """Run one operation once (setup, execute, validate, teardown) and show its result."""
    scenario, meta = SCENARIOS[operation_id], META[operation_id]
    ctx = scenario.setup()
    try:
        started = time.perf_counter()
        result = scenario.execute(ctx)
        elapsed = (time.perf_counter() - started) * 1000
        passed = bool(scenario.validate(ctx, result))
    finally:
        scenario.teardown(ctx)
    count = len(result) if isinstance(result, list) else None
    SUMMARY[operation_id] = {'operation_id': operation_id, 'part2': meta.part2_query,
                             'kind': meta.kind, 'description': meta.description,
                             'requirements': ', '.join(meta.requirement_ids), 'rows': count,
                             'time_ms': round(elapsed, 1), 'correct': passed}
    head = f'**{operation_id}**' + (f' (Part 2 {meta.part2_query})' if meta.part2_query else '')
    display(Markdown(f'{head}: {meta.description}. '
                     + (f'{count:,} rows, ' if count is not None else '')
                     + f'{elapsed:.1f} ms, validation **{"PASSED" if passed else "FAILED"}**'))
    return as_table(result, rows)

### PF-01: Orders and trips (create, read)

In [ ]:
run('PF-01-Q01')

In [ ]:
run('PF-01-Q02')

In [ ]:
run('PF-01-Q03')

### PF-02: Driver assignments

In [ ]:
run('PF-02-Q01')

In [ ]:
run('PF-02-Q02')

In [ ]:
run('PF-02-Q03')

In [ ]:
run('PF-02-Q04')

### PF-03: Service status

In [ ]:
run('PF-03-Q01')

In [ ]:
run('PF-03-Q02')

### PF-04: Payments

In [ ]:
run('PF-04-Q01')

In [ ]:
run('PF-04-Q02')

In [ ]:
run('PF-04-Q03')

### PF-05: Outlet search by area

In [ ]:
run('PF-05-Q01')

### PF-06: Service areas and outlets

In [ ]:
run('PF-06-Q01')

In [ ]:
run('PF-06-Q02')

### PF-07: Deletion

In [ ]:
run('PF-07-Q01')

In [ ]:
run('PF-07-Q02')

### PF-08: Driver and vehicle performance

In [ ]:
run('PF-08-Q01')

In [ ]:
run('PF-08-Q02')

In [ ]:
run('PF-08-Q03')

### PF-09: Activity and demand

In [ ]:
run('PF-09-Q01')

In [ ]:
run('PF-09-Q02')

### PF-10: Payments and revenue

In [ ]:
run('PF-10-Q01')

In [ ]:
run('PF-10-Q02')

In [ ]:
run('PF-10-Q03')

### Summary

In [ ]:
summary = pd.DataFrame(SUMMARY.values()).sort_values('operation_id').reset_index(drop=True)
print(f"{summary['correct'].sum()}/{len(summary)} operations returned the expected result")
summary

## 5. Query the database yourself

The client wraps the CouchDB HTTP API one request per call, so any request can be tried here.
`DB` holds the database names, e.g. `DB['orders'] == 'grab_orders'`.

In [ ]:
# One order document: subtype, payment, trips and assignments together.
first_order = admin.all_docs(DB['orders'], startkey='ORD', endkey='ORD\ufff0', limit=1)['rows'][0]['id']
admin.get_doc(DB['orders'], first_order)

In [ ]:
# The design documents and their views (section 2.4.1).
pd.DataFrame([{'database': settings.db(name), 'design_doc': ddoc['_id'], 'view': view,
               'reduce': spec.get('reduce', '')}
              for name in DATABASES for ddoc in design.view_ddocs(name)
              for view, spec in ddoc['views'].items()])

In [ ]:
# A view query: trips per service area (grouped _count), like R5 before joining area names.
pd.DataFrame(admin.view(DB['orders'], 'analytics', 'trips_by_start_area', group=True)['rows'])

In [ ]:
# A Mango query and how CouchDB executes it: the index it uses and how much it reads.
query = {
    'selector': {'location.service_area_id': 'SAR0000007', 'outlet_status': 'OPEN'},
    'sort': [{'location.service_area_id': 'asc'}, {'outlet_status': 'asc'},
             {'outlet_name': 'asc'}, {'merchant_outlet_id': 'asc'}],
    'fields': ['merchant_outlet_id', 'outlet_name'],
    'use_index': ['mango-outlets', 'open-outlets-by-area'],
    'limit': 5, 'execution_stats': True,
}
found = admin.find(DB['geo'], query)
explain = admin.json('POST', f"/{DB['geo']}/_explain", body=query)
print('index used:', explain['index']['ddoc'], explain['index']['name'])
print('execution stats:', found.get('execution_stats'))
pd.DataFrame(found['docs'])

In [ ]:
# Write your own query here, e.g.:
# admin.view(DB['orders'], 'ops', 'by_customer', startkey=['CUS0000001'], endkey=['CUS0000001', {}])

## 6. Benchmark (optional)

Runs `benchmarks/run.py` with the CouchDB adapter: one warm-up and three measured runs per
operation, timing only `execute()`. Results go to `benchmarks/results/` (`runs.csv`,
`results.json`); copy them to `benchmarks/published/` once they are final.

In [ ]:
if RUN_BENCHMARK:
    from benchmarks import run as runner
    dataset_size = json.loads((DATA_DIR / 'manifest.json').read_text())['tables']['service_orders']['rows'] \
        if (DATA_DIR / 'manifest.json').is_file() else sum(1 for _ in open(DATA_DIR / 'service_orders.csv')) - 1
    runner.main(['--adapter', 'benchmarks.adapters.couchdb', '--dataset-size', str(dataset_size)])

results_file = ROOT / 'benchmarks' / 'results' / 'results.json'
if results_file.is_file():
    results = json.loads(results_file.read_text())['results']
    table = pd.DataFrame([{
        'operation_id': r['operation_id'], 'part2': META[r['operation_id']].part2_query,
        'scenario': r['scenario'], 'correct': r['validation']['passed'],
        'runs_ms': r['measurements_ms'], 'mean_ms': r['mean_ms'], 'median_ms': r['median_ms'],
    } for r in results if 'operation_id' in r and r.get('dbms') == 'couchdb'])
    display(table.sort_values('operation_id').reset_index(drop=True))
else:
    print('No benchmark results yet: set RUN_BENCHMARK = True.')